In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "qc", "prod"])
dbutils.widgets.text("storage_account", "")   # ← change this

env  = dbutils.widgets.get("env")
acct = dbutils.widgets.get("storage_account")
layers  = ["bronze", "silver", "gold"]
schemas = ["africa", "america", "asia", "europe", "middle_east", "common"]
for layer in layers:
    catalog  = f"{env}_{layer}"
    location = f"abfss://{layer}@{acct}.dfs.core.windows.net/"
    spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS {catalog}
        MANAGED LOCATION '{location}'
        COMMENT '{env.upper()} {layer} layer - retail sales'
    """)
    for s in schemas:
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS {catalog}.{s}
            COMMENT '{layer} data for {s.replace('_', ' ')}'
        """)



# Folder for raw file uploads (CSV/JSON) later
spark.sql(f"CREATE VOLUME IF NOT EXISTS {env}_bronze.common.landing")
print("✔ volume dev_bronze.common.landing")


In [0]:

%sql
SHOW CATALOGS LIKE 'dev_*';


SHOW SCHEMAS IN dev_silver;


CREATE TABLE dev_bronze.common.smoke_test AS SELECT 1 AS id, current_timestamp() AS ts;
SELECT * FROM dev_bronze.common.smoke_test;


In [0]:
%sql
DROP TABLE dev_bronze.common.smoke_test;
